# 9주차 학생용 실습 — 대규모 센서 데이터와 간단한 불량 예측

오늘의 질문: **센서 값만 보고 합격/불합격을 미리 알 수 있을까?**

⚠️ 이 데이터의 열 이름(Chamber_Temperature_edu, 주요센서_A 등)은 실제 센서의 물리적 의미를
확인한 것이 아니라, 학습을 돕기 위한 **교육용 가상 별칭**입니다. `_edu`가 붙은 이름과
'주요센서_A/B'는 실제 장비 의미를 단정하지 않습니다.

`# TODO` 표시가 있는 셀만 직접 채워 넣으면 됩니다.

## 1단계. 데이터 불러오기

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week09/week09_fab_beginner.csv")
df.head()

## 2단계. 데이터 크기와 정상/이상 비율 확인하기
`검사결과`: 0=정상/합격, 1=이상/불합격

In [ ]:
print(df.shape)
print(df["검사결과"].value_counts())

**질문**: 정상과 이상의 개수 차이가 매우 큽니다. 이런 상태를 무엇이라고 부를까요? (힌트: 데이터 불균형)

## 3단계. 결측값 확인 및 처리

In [ ]:
feature_cols = [
    "Chamber_Temperature_edu", "Chamber_Pressure_edu", "Gas_Flow_edu", "RF_Power_edu",
    "Vacuum_Level_edu", "Cooling_Water_Temperature_edu", "Vibration_edu", "Process_Time_edu",
    "주요센서_A", "주요센서_B",
]
print(df[feature_cols].isna().sum())

# TODO: 빈칸을 채워 결측값을 각 열의 평균으로 채우세요.
df[feature_cols] = df[feature_cols].____(df[feature_cols].mean())

## 4단계. 학습 데이터와 테스트 데이터 나누기

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df["검사결과"]

# TODO: 빈칸에 test_size=0.3, random_state=42, stratify=y 를 채우세요.
X_train, X_test, y_train, y_test = train_test_split(X, y, ____)
print("학습 데이터:", X_train.shape, " 테스트 데이터:", X_test.shape)

## 5단계. 의사결정나무 모델 학습하기

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
print("학습 완료")

## 6단계. 예측하고 정확도 확인하기

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
# TODO: 빈칸을 채워 정확도를 계산하세요.
acc = ____(y_test, y_pred)
print(f"정확도: {acc*100:.2f}%")

## 7단계. '모두 정상'이라고만 예측했을 때의 정확도와 비교하기

In [ ]:
naive_acc = (y_test == 0).mean() * 100
print(f"'모두 정상'이라고만 예측했을 때 정확도: {naive_acc:.2f}%")

**질문**: 모델의 정확도와 '모두 정상' 예측의 정확도를 비교해보세요. 정확도만으로 모델이
좋다고 말할 수 있을까요? 왜 그럴까요?

## 8단계. 혼동행렬로 자세히 확인하기

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("혼동행렬 (행=실제, 열=예측, 순서 0=정상,1=이상):")
print(cm)

**질문**: 혼동행렬에서 '실제 이상(1)을 이상(1)이라고 맞춘 개수'는 몇 개인가요?
이 숫자가 왜 정확도보다 더 중요한 정보일 수 있을까요?

## 9단계(도전). 어떤 센서가 예측에 가장 중요했는지 확인하기

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False)

⚠️ **주의**: 위 결과는 이 모델이 예측할 때 어떤 열을 더 많이 활용했는지 보여줄 뿐,
실제 불량의 '원인'을 증명하는 것이 아닙니다.

## 10단계. 오늘의 분석을 한 문장으로 정리하기

> (예시) 모델의 정확도는 ___%였지만, '모두 정상'이라고 예측해도 ___%의 정확도가 나온다.
> 따라서 정확도만으로는 모델의 성능을 판단하기 어렵고, 혼동행렬을 함께 봐야 한다.